# Linear Discriminant Analysis (LDA)

## LDA vs PCA: The Key Difference

Both PCA and LDA reduce dimensions. But they optimise for completely different things:

| | PCA | LDA |
|-|-----|-----|
| **Goal** | Maximise variance in the data | Maximise class separability |
| **Uses labels?** | No — unsupervised | Yes — supervised |
| **Finds directions that...** | spread data out the most | push classes apart |
| **Maximum components** | min(n_samples, n_features) - 1 | n_classes - 1 |

---

## The Intuition

PCA does not know what your classes are. It just finds the directions of highest variance in the data — which may or may not help a classifier.

LDA **uses the class labels** to find the directions that maximally separate the classes. It simultaneously:
1. **Maximises** the distance between class means (between-class scatter)
2. **Minimises** the spread within each class (within-class scatter)

```
PCA projection:                LDA projection:

  Class A: ooo         →       Class A: ooo
  Class B: xxx                 Class B:           xxx
  Both overlapping             Cleanly separated
```

---

## The Mathematics (Intuition)

LDA solves for the projection matrix $W$ that maximises the **Fisher criterion**:

$$J(W) = \frac{W^T S_B W}{W^T S_W W}$$

Where:
- $S_B$ = **between-class scatter matrix** — measures how far apart the class means are
- $S_W$ = **within-class scatter matrix** — measures how spread out samples are within each class

Maximising this ratio = maximising separation while minimising overlap.

---

## When to Use LDA Over PCA

| Situation | Choose |
|-----------|--------|
| You have class labels and want maximum separability | **LDA** |
| You want to visualise class clusters | **LDA** |
| You are doing unsupervised or exploratory analysis | **PCA** |
| You have few samples per class (LDA can overfit) | **PCA** |
| Classes are not linearly separable | **Kernel PCA** |

**Maximum components:** LDA produces at most $K - 1$ components where $K$ is the number of classes. With 3 wine classes, LDA gives maximum 2 components — exactly what we use here.

---

## What We Will Build

Same setup as the PCA notebook (Wine dataset, Logistic Regression classifier), but using LDA for dimensionality reduction. The comparison will show whether LDA's class-aware approach gives better separation than PCA's variance-maximisation.

## Step 1: Import Libraries

Same three libraries as PCA — LDA is available from `sklearn.discriminant_analysis`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Step 2: Load the Dataset

Same Wine dataset as the PCA notebook: 178 samples, 13 chemical features, 3 wine classes.

Using the same dataset as PCA lets us **directly compare** the two dimensionality reduction methods — same data, same downstream classifier, only the reduction technique changes.

In [ ]:
dataset = pd.read_csv('Wine.csv')
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

## Step 3: Train/Test Split

Same 80/20 split. The split must come before LDA — the discriminant directions must be learned from training data only.

**Unlike PCA, LDA uses the labels `y_train` during fitting.** This is what makes it supervised. The labels are used to compute between-class and within-class scatter matrices.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)

## Step 4: Feature Scaling

LDA also requires feature scaling, for the same reason as PCA: the scatter matrices ($S_B$ and $S_W$) are built from covariances. Features on vastly different scales will dominate the scatter calculation regardless of their true discriminative power.

StandardScaler ensures all features contribute equally to the scatter matrices.

In [ ]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

## Step 5: Apply LDA

```python
lda = LDA(n_components=2)
X_train = lda.fit_transform(X_train, y_train)  # Note: y_train is required
X_test  = lda.transform(X_test)                # No labels needed for transform
```

**The critical difference from PCA:** `fit_transform` takes both `X_train` **and** `y_train`. LDA needs the class labels to compute its scatter matrices — without them it cannot identify which directions maximise class separation.

The result is the same shape as PCA: X is reduced to 2 columns (LD1, LD2 — Linear Discriminants 1 and 2).

**Why exactly 2 components?** With 3 wine classes, the maximum number of LDA components is 3 - 1 = 2. This is a hard mathematical limit — LDA cannot produce more components than classes minus one.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
lda = LDA(n_components = 2)
X_train = lda.fit_transform(X_train, y_train)
X_test = lda.transform(X_test)

## Step 6: Train Logistic Regression on LDA-Reduced Features

The classifier receives 2 features (LD1, LD2) instead of 13.

Because LDA specifically optimised the 2D projection for class separation, the logistic regression has an easier classification problem than it would with 2 random PCA components.

In [ ]:
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression(random_state = 0)
classifier.fit(X_train, y_train)

## Step 7: Evaluate — Confusion Matrix

**100% accuracy on the test set** with just 2 LDA features vs 13 original features.

Compare this to the PCA result (~97% accuracy). LDA achieved perfect classification on the same dataset — because its 2 components were specifically chosen to maximally separate the 3 wine classes.

**Why does LDA outperform PCA here?**

The 2 principal components PCA selected capture the most variance — but maximum variance is not the same as maximum class discriminability. The direction of most variance might contain a mix of all three classes. LDA ignores overall variance and directly optimises for the projection that pushes the class clouds apart.

**A word of caution on 100% accuracy:**

Perfect test accuracy on 36 samples (20% of 178) is not a guarantee of real-world performance. With a small test set, 100% can happen by chance. Cross-validation over all 178 samples would give a more reliable estimate.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = classifier.predict(X_test)
cm = confusion_matrix(y_test, y_pred)
print(cm)
accuracy_score(y_test, y_pred)

## Step 8: Visualise the Training Set

The axes are now **LD1 and LD2** — Linear Discriminants — not PCs.

You should see extremely clean cluster separation compared to the PCA visualisation. The three wine classes should be almost completely non-overlapping in 2D LDA space, because that is exactly what LDA was optimised to produce.

This is the power of using label information for dimensionality reduction: LDA finds a 2D view of the data that makes the classification problem as easy as possible.

In [ ]:
from matplotlib.colors import ListedColormap
X_set, y_set = X_train, y_train
X1, X2 = np.meshgrid(np.arange(start = X_set[:, 0].min() - 1, stop = X_set[:, 0].max() + 1, step = 0.01),
                     np.arange(start = X_set[:, 1].min() - 1, stop = X_set[:, 1].max() + 1, step = 0.01))
plt.contourf(X1, X2, classifier.predict(np.array([X1.ravel(), X2.ravel()]).T).reshape(X1.shape),
             alpha = 0.75, cmap = ListedColormap(('red', 'green', 'blue')))
plt.xlim(X1.min(), X1.max())
plt.ylim(X2.min(), X2.max())
for i, j in enumerate(np.unique(y_set)):
    plt.scatter(X_set[y_set == j, 0], X_set[y_set == j, 1],
                c = ListedColormap(('red', 'green', 'blue'))(i), label = j)
plt.title('Logistic Regression (Training set)')
plt.xlabel('LD1')
plt.ylabel('LD2')
plt.legend()
plt.show()

## Step 9: Visualise the Test Set

The near-perfect separation on the test set confirms that LDA found genuinely discriminative directions, not just ones that memorised the training data.

**LDA vs PCA — Summary:**

| | PCA | LDA |
|-|-----|-----|
| Accuracy (this dataset) | ~97% | ~100% |
| Visual separation | Good | Excellent |
| Requires class labels | No | Yes |
| Risk of overfitting | Lower | Higher (with few samples per class) |

Use LDA when you have labelled data and want to optimise for classification. Use PCA when labels are unavailable, you are exploring the data, or you have too few samples per class for LDA to be reliable.

In [ ]:
from matplotlib.colors import ListedColormap
X_set, y_set = X_test, y_test
X1, X2 = np.meshgrid(np.arange(start = X_set[:, 0].min() - 1, stop = X_set[:, 0].max() + 1, step = 0.01),
                     np.arange(start = X_set[:, 1].min() - 1, stop = X_set[:, 1].max() + 1, step = 0.01))
plt.contourf(X1, X2, classifier.predict(np.array([X1.ravel(), X2.ravel()]).T).reshape(X1.shape),
             alpha = 0.75, cmap = ListedColormap(('red', 'green', 'blue')))
plt.xlim(X1.min(), X1.max())
plt.ylim(X2.min(), X2.max())
for i, j in enumerate(np.unique(y_set)):
    plt.scatter(X_set[y_set == j, 0], X_set[y_set == j, 1],
                c = ListedColormap(('red', 'green', 'blue'))(i), label = j)
plt.title('Logistic Regression (Test set)')
plt.xlabel('LD1')
plt.ylabel('LD2')
plt.legend()
plt.show()